# Week 6 — Apache Spark Assignment
## Spark Architecture, Transformations, Schema Handling & Optimized File Formats

**Author:** Data Engineering Cohort — Week 6 Submission
**Tool:** PySpark (Apache Spark 3.5.x) on Jupyter Notebook

---

### Overview

This notebook is a complete, self-contained walkthrough of core Apache Spark concepts required for efficient large-scale data processing. It combines **conceptual explanations** (architecture, lazy evaluation, DAG, shuffle, predicate pushdown) with **hands-on PySpark code** (reading CSV/Parquet, schema handling, filtering, transformations, and building a full read → transform → filter → write pipeline).

Every question Q1–Q15 from the assignment brief is answered in its own section, with markdown explanation followed by executable, well-commented PySpark code and real output.


## Learning Objectives

By the end of this notebook you will be able to:

1. Describe the roles of the **Driver**, **Cluster Manager**, and **Executors** in a Spark application.
2. Explain Spark's **execution modes** (local, client, cluster).
3. Explain **Lazy Evaluation** and how Spark builds a **DAG (lineage graph)**.
4. Read data from **CSV** and **Parquet** with explicit and inferred schemas.
5. Perform **filtering** and **column selection** on DataFrames.
6. **Modify DataFrames**: rename columns, cast types, add derived columns.
7. Distinguish between **transformations** and **actions**.
8. Explain **wide transformations** and the **shuffle** process.
9. Explain **Predicate Pushdown** and why it matters for Parquet.
10. Compare **CSV vs Parquet** performance characteristics.
11. Handle **null values** efficiently.
12. Build a complete **data pipeline**: Read → Transform → Filter → Write.
13. Write output in **both CSV and Parquet** formats.
14. Apply Spark **best practices** for large datasets (e.g., avoiding `collect()`).


## 1. Environment Setup

We use **PySpark** as the only processing engine, as required by the assignment. The cell below verifies the installed PySpark version and Java runtime (Spark requires a JVM under the hood).


In [1]:
# Verify environment
import pyspark
import sys

print(f"Python version : {sys.version.split()[0]}")
print(f"PySpark version: {pyspark.__version__}")


Python version : 3.12.3
PySpark version: 3.5.1


## 2. Required Imports

All the PySpark modules used throughout this notebook are imported up-front, following best practice of keeping imports centralized at the top of the notebook.


In [2]:
# Core Spark imports
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType
)
import time
import os
import shutil


## 3. SparkSession Creation

The **`SparkSession`** is the unified entry point to all Spark functionality (SQL, DataFrame API, streaming, etc.). In a real cluster deployment, `master()` would point to a YARN/Kubernetes/Standalone cluster URL (e.g. `yarn`, `k8s://...`, `spark://host:7077`). Here we use `local[*]` to use all available cores on this machine for development/testing — this is Spark's **local execution mode**.

A few configuration tweaks are applied:
- `spark.sql.shuffle.partitions` is reduced from the 200 default since our sample dataset is small (avoids excessive small-file/partition overhead).
- `spark.ui.showConsoleProgress` is disabled to keep notebook output clean.


In [3]:
# Create (or get existing) SparkSession
spark = (
    SparkSession.builder
    .appName("Week6_Apache_Spark_Assignment")
    .master("local[*]")                       # local mode: driver + executors on this machine
    .config("spark.sql.shuffle.partitions", "8")  # tuned down for small sample data
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()

)

spark.sparkContext.setLogLevel("ERROR")  # reduce log noise in notebook output

print("SparkSession created successfully")
print(f"Spark version : {spark.version}")
print(f"Master         : {spark.sparkContext.master}")
print(f"App Name       : {spark.sparkContext.appName}")


26/06/30 15:25:23 WARN Utils: Your hostname, vm resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/06/30 15:25:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/06/30 15:25:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession created successfully
Spark version : 3.5.1
Master         : local[*]
App Name       : Week6_Apache_Spark_Assignment


## 4. Sample Dataset

A realistic e-commerce **orders dataset** (`data/source.csv`) is provided in the repository so every question can be executed end-to-end without depending on an external data source. It contains 20 rows with the following columns:

| Column | Description |
|---|---|
| `order_id` | Unique order identifier |
| `product_id`, `product_name` | Product identifiers |
| `category` | Product category (Electronics, Furniture, Stationery) |
| `old_name` | Legacy customer-name column (used for the rename exercise, Q6) |
| `price` | Unit price (stored as string in CSV — used for the cast exercise, Q6) |
| `base_price` | Base price before tax (used to derive `final_price`, Q10) |
| `quantity` | Units ordered |
| `status` | Order status: Completed / Pending / Cancelled |
| `amount` | Total order amount |
| `region` | Delivery region |
| `priority` | Order priority |
| `user_id` | Customer identifier — **contains intentional nulls** for the null-handling exercise (Q12) |

Let's preview the raw file before loading it into Spark.


In [4]:
# Peek at the raw CSV file (plain Python, not Spark — just to inspect structure)
with open("../data/source.csv", "r") as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i == 5:
            break


order_id,product_id,product_name,category,old_name,price,base_price,quantity,status,amount,region,priority,user_id
1001,P001,Wireless Mouse,Electronics,Customer A,599.0,500.0,2,Completed,1198.0,North,High,U001
1002,P002,Office Chair,Furniture,Customer B,4500.0,4000.0,1,Pending,4500.0,South,Low,U002
1003,P003,Bluetooth Speaker,Electronics,Customer C,1999.0,1700.0,1,Completed,1999.0,North,Medium,U003
1004,P004,Notebook Set,Stationery,Customer D,150.0,120.0,5,Completed,750.0,East,Low,U004
1005,P005,Gaming Laptop,Electronics,Customer E,75000.0,65000.0,1,Completed,75000.0,West,High,U005


We will also persist a **Parquet copy** of this same dataset (`data/sample_parquet/`) later in the notebook so that the Parquet-specific questions (Q3 comparison, Q9, Q12) have a real columnar file to operate on.

---
## Q1 — Spark Architecture: Driver, Cluster Manager, Executor

**Question:** Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.

### Explanation

| Component | Role |
|---|---|
| **Driver** | The process that runs the `main()` function of the Spark application. It creates the `SparkSession`/`SparkContext`, converts user code into a **logical plan → physical plan → DAG of stages and tasks**, schedules tasks onto executors, and collects/aggregates results. It is the "brain" of the application — if it fails, the whole application fails. |
| **Cluster Manager** | A pluggable component responsible for **resource allocation** across the cluster. Spark supports several cluster managers: **Standalone**, **YARN**, **Kubernetes**, and **Mesos** (legacy). It negotiates with the cluster to launch executor processes on worker nodes on behalf of the driver. It does *not* execute Spark tasks itself — it only allocates resources (CPU/memory/containers). |
| **Executor** | A JVM process launched on a worker node that actually **runs tasks** (the unit of work derived from RDD/DataFrame partitions) and **caches data** in memory/disk when requested (`.cache()`/`.persist()`). Each executor runs many tasks concurrently in its own threads and reports status/results back to the Driver. Executors live for the duration of the application (in normal, non-dynamic-allocation mode). |

**Flow:** Driver → asks Cluster Manager for resources → Cluster Manager launches Executors on Worker nodes → Driver sends Tasks to Executors → Executors process partitions and return results/status to the Driver.

In our notebook, since we use `local[*]`, the Driver and Executors run as threads within the **same single JVM process** — useful for development, but a real production job submits to YARN/K8s/Standalone with separate executor JVMs on separate worker machines.


In [5]:
# Inspect the live Spark application's resource configuration
sc = spark.sparkContext

print("Driver host        :", sc.getConf().get("spark.driver.host"))
print("Master / Cluster Mgr:", sc.master)
print("Default Parallelism :", sc.defaultParallelism, "(≈ number of cores Executors will use)")
print("Application ID      :", sc.applicationId)
print("Application Name    :", sc.appName)

# In local mode, driver and executor share one JVM -> only 1 'executor' (the driver itself) is reported
print("\nExecutor / driver memory status:")
print(sc._jsc.sc().getExecutorMemoryStatus())


Driver host        : 192.0.2.2
Master / Cluster Mgr: local[*]
Default Parallelism : 1 (≈ number of cores Executors will use)
Application ID      : local-1782833125648
Application Name    : Week6_Apache_Spark_Assignment

Executor / driver memory status:
Map(192.0.2.2:33827 -> (434031820,434031820))


---
## Q2 — Lazy Evaluation

**Question:** How does Spark's Lazy Evaluation strategy improve performance when chain-processing large datasets?

### Explanation

Spark **transformations** (`select`, `filter`, `withColumn`, `groupBy`, etc.) are **not executed immediately**. Instead, Spark builds up a **logical plan** describing *what* should happen. Execution is triggered only when an **action** (`show()`, `count()`, `write()`, `collect()`) is called.

Benefits of this laziness:

1. **Query optimization** — The Catalyst optimizer can look at the *entire* chain of transformations at once (instead of one operation at a time) and reorder/merge/eliminate operations — e.g., pushing filters earlier, combining multiple `select`s, eliminating unused columns (column pruning).
2. **Avoids unnecessary computation** — If a later step makes earlier output irrelevant (e.g., a `LIMIT`), Spark can skip processing data that was never going to be used.
3. **Fewer passes over data** — Instead of materializing every intermediate DataFrame (which is expensive for huge datasets), Spark fuses compatible transformations into a single physical stage executed in one pass.
4. **Pushdown to source** — Filters/projections can be pushed all the way down to the data source (e.g., Parquet predicate pushdown, JDBC pushdown), reducing the amount of data actually read from disk/network.

In short: laziness lets Spark see the **whole picture** before doing any real work, so it can choose the most efficient physical execution plan rather than executing each line naively as it's written.


In [6]:
# Demonstrate lazy evaluation: build a long transformation chain (NOTHING executes yet)
df_orders_raw = spark.read.csv("../data/source.csv", header=True, inferSchema=True)

lazy_chain = (
    df_orders_raw
    .filter(F.col("status") == "Completed")
    .select("order_id", "product_name", "category", "amount")
    .withColumn("amount_with_tax", F.col("amount") * 1.18)
    .filter(F.col("amount_with_tax") > 1000)
)

print("Transformation chain built — no Spark job has run yet.")
print("Logical plan (not yet executed):")
lazy_chain.explain(mode="simple")

# The job is only triggered now, by an ACTION:
print("\nTriggering execution via the action .show()  ...")
lazy_chain.show(5)


Transformation chain built — no Spark job has run yet.
Logical plan (not yet executed):
== Physical Plan ==
*(1) Project [order_id#17, product_name#19, category#20, amount#26, (amount#26 * 1.18) AS amount_with_tax#48]
+- *(1) Filter (((isnotnull(status#25) AND isnotnull(amount#26)) AND (status#25 = Completed)) AND ((amount#26 * 1.18) > 1000.0))
   +- FileScan csv [order_id#17,product_name#19,category#20,status#25,amount#26] Batched: false, DataFilters: [isnotnull(status#25), isnotnull(amount#26), (status#25 = Completed), ((amount#26 * 1.18) > 1000.0)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/claude/Week6-Apache-Spark/data/source.csv], PartitionFilters: [], PushedFilters: [IsNotNull(status), IsNotNull(amount), EqualTo(status,Completed)], ReadSchema: struct<order_id:int,product_name:string,category:string,status:string,amount:double>



Triggering execution via the action .show()  ...


+--------+-----------------+-----------+-------+------------------+
|order_id|     product_name|   category| amount|   amount_with_tax|
+--------+-----------------+-----------+-------+------------------+
|    1001|   Wireless Mouse|Electronics| 1198.0|1413.6399999999999|
|    1003|Bluetooth Speaker|Electronics| 1999.0|2358.8199999999997|
|    1005|    Gaming Laptop|Electronics|75000.0|           88500.0|
|    1007|      USB-C Cable|Electronics|  897.0|           1058.46|
|    1009|      Smart Watch|Electronics| 8999.0|          10618.82|
+--------+-----------------+-----------+-------+------------------+
only showing top 5 rows



---
## Q3 — Read CSV with Header and Inferred Schema

**Question:** Write a Spark command to read a CSV file located at `data/source.csv`, ensuring the first row is treated as a header and `inferSchema` is enabled.


In [7]:
# Read CSV with header=True and inferSchema=True
df_csv = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("../data/source.csv")
)

print("Schema inferred by Spark:")
df_csv.printSchema()

print(f"Row count: {df_csv.count()}")
df_csv.show(5)


Schema inferred by Spark:
root
 |-- order_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- old_name: string (nullable = true)
 |-- price: double (nullable = true)
 |-- base_price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- user_id: string (nullable = true)



Row count: 20


+--------+----------+-----------------+-----------+----------+-------+----------+--------+---------+-------+------+--------+-------+
|order_id|product_id|     product_name|   category|  old_name|  price|base_price|quantity|   status| amount|region|priority|user_id|
+--------+----------+-----------------+-----------+----------+-------+----------+--------+---------+-------+------+--------+-------+
|    1001|      P001|   Wireless Mouse|Electronics|Customer A|  599.0|     500.0|       2|Completed| 1198.0| North|    High|   U001|
|    1002|      P002|     Office Chair|  Furniture|Customer B| 4500.0|    4000.0|       1|  Pending| 4500.0| South|     Low|   U002|
|    1003|      P003|Bluetooth Speaker|Electronics|Customer C| 1999.0|    1700.0|       1|Completed| 1999.0| North|  Medium|   U003|
|    1004|      P004|     Notebook Set| Stationery|Customer D|  150.0|     120.0|       5|Completed|  750.0|  East|     Low|   U004|
|    1005|      P005|    Gaming Laptop|Electronics|Customer E|75000.0

**Best practice note:** `inferSchema=True` requires Spark to make an *extra pass* over the data just to determine column types, which adds overhead — fine for small/medium files or exploration, but for production pipelines on large files it's better to **define an explicit `StructType` schema** (shown below) so Spark can read in a single pass and guarantee type correctness.

In [8]:
# Production-style alternative: explicit schema (single pass, no inference overhead)
explicit_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("old_name", StringType(), True),
    StructField("price", StringType(), True),       # kept as String on purpose -> cast later in Q6
    StructField("base_price", DoubleType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("region", StringType(), True),
    StructField("priority", StringType(), True),
    StructField("user_id", StringType(), True),
])

df_orders = spark.read.csv("../data/source.csv", header=True, schema=explicit_schema)
df_orders.printSchema()


root
 |-- order_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- old_name: string (nullable = true)
 |-- price: string (nullable = true)
 |-- base_price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- user_id: string (nullable = true)



---
## Q4 — CSV vs Parquet: Row-based vs Columnar Storage

**Question:** What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar), and why does it matter for performance?

### Explanation

| Aspect | CSV (row-based) | Parquet (columnar) |
|---|---|---|
| **Layout** | Stores data **row by row**, all columns of a record together, as plain text. | Stores data **column by column**; all values of one column are stored contiguously, in binary, with per-column encoding. |
| **Schema** | No embedded schema — types must be inferred or declared separately. | Schema is **embedded in the file** (self-describing). |
| **Compression** | Limited — text compresses generically (gzip), not type-aware. | Excellent — similar values stored together compress far better (dictionary/RLE encoding), often 4–10x smaller than CSV. |
| **I/O for analytics** | Must read **every column of every row**, even if only 2 of 20 columns are needed. | Can read **only the columns requested** (column pruning) — drastically reduces I/O. |
| **Predicate pushdown** | Not possible — text must be parsed before any filtering. | Supported — min/max statistics per row-group let Spark **skip entire blocks** that can't match a filter (see Q9). |
| **Splittability / parallelism** | Splittable, but parsing text (type conversion) is CPU-heavy. | Splittable by row-group, and reading binary columnar data is much faster to deserialize. |
| **Best use case** | Interop with external systems, human-readability, ad-hoc exports. | Internal pipeline storage, repeated analytical reads, big data at scale. |

**Why it matters:** Most analytical queries touch a small subset of columns (`SELECT category, amount FROM orders`) and often filter rows (`WHERE status = 'Completed'`). Columnar storage lets Spark skip irrelevant columns and irrelevant row-groups entirely — this is the single biggest reason Parquet dramatically outperforms CSV for big-data analytics.


In [9]:
# Quick visual proof: file size comparison after we write the same data as CSV vs Parquet (full demo in Q12/pipeline section)
# (executed fully later — placeholder explanation cell, see Section 'CSV vs Parquet Performance Comparison')
print("See the dedicated 'CSV vs Parquet Performance Comparison' benchmark later in this notebook (after Q12).")


See the dedicated 'CSV vs Parquet Performance Comparison' benchmark later in this notebook (after Q12).


---
## Q5 — Select Specific Columns with a Filter

**Question:** Given a DataFrame `df`, write a query to select the columns `product_id` and `price` where the category is "Electronics".


In [10]:
df = df_orders  # alias, as referenced in the question

electronics_price = df.select("product_id", "price").filter(F.col("category") == "Electronics")

print(f"Rows matching category == 'Electronics': {electronics_price.count()}")
electronics_price.show(10)


Rows matching category == 'Electronics': 10


+----------+-------+
|product_id|  price|
+----------+-------+
|      P001|  599.0|
|      P003| 1999.0|
|      P005|75000.0|
|      P007|  299.0|
|      P009| 8999.0|
|      P011|18999.0|
|      P013| 1299.0|
|      P015| 6499.0|
|      P017| 1499.0|
|      P019|22999.0|
+----------+-------+



---
## Q6 — Rename Column & Cast Data Type

**Question:** Rename the column `old_name` to `new_name` and cast the `price` column from String to Double.


In [11]:
df_renamed_casted = (
    df_orders
    .withColumnRenamed("old_name", "new_name")
    .withColumn("price", F.col("price").cast(DoubleType()))
)

print("Updated schema:")
df_renamed_casted.printSchema()

df_renamed_casted.select("order_id", "new_name", "price").show(5)


Updated schema:
root
 |-- order_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- new_name: string (nullable = true)
 |-- price: double (nullable = true)
 |-- base_price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- user_id: string (nullable = true)



+--------+----------+-------+
|order_id|  new_name|  price|
+--------+----------+-------+
|    1001|Customer A|  599.0|
|    1002|Customer B| 4500.0|
|    1003|Customer C| 1999.0|
|    1004|Customer D|  150.0|
|    1005|Customer E|75000.0|
+--------+----------+-------+
only showing top 5 rows



**Note:** `withColumnRenamed` and `withColumn(...).cast(...)` are both **narrow transformations** — they operate independently per-partition with no data movement (no shuffle) between executors, so they are very cheap operations.

---
## Q7 — Lineage Graph (DAG) and Fault Tolerance

**Question:** How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails?

### Explanation

Every RDD/DataFrame in Spark remembers **how it was derived** from its parent(s) — the sequence of transformations applied (`map`, `filter`, `join`, etc.). This recorded history is the **lineage graph**, represented internally as a **Directed Acyclic Graph (DAG)** of stages and tasks.

When a **worker node / executor fails** mid-job:

1. Spark detects the lost executor (heartbeat timeout) and marks its partitions' tasks/output as lost.
2. Because each partition's lineage is known, Spark **recomputes only the lost partitions** by re-running the transformations from the nearest available data (original source, or a cached/checkpointed ancestor) — it does **not** need to restart the whole job from scratch.
3. The recomputed partitions are reassigned to a healthy executor, and the job continues.

This is fundamentally different from systems that rely on full data replication for fault tolerance — Spark instead relies on **recomputation via lineage**, which is cheaper for large datasets since only the affected partitions are redone. (For very expensive lineages, `.checkpoint()` can be used to truncate lineage and persist materialized data to durable storage, avoiding deep recomputation chains.)


In [12]:
# Visualize a DAG / lineage via the logical & physical plan of a multi-step chain
pipeline_demo = (
    df_orders
    .filter(F.col("status") == "Completed")
    .groupBy("category")
    .agg(F.sum("amount").alias("total_amount"))
    .orderBy(F.desc("total_amount"))
)

print("DAG / physical plan showing the stages Spark will execute:")
pipeline_demo.explain(mode="formatted")


DAG / physical plan showing the stages Spark will execute:
== Physical Plan ==
AdaptiveSparkPlan (9)
+- Sort (8)
   +- Exchange (7)
      +- HashAggregate (6)
         +- Exchange (5)
            +- HashAggregate (4)
               +- Project (3)
                  +- Filter (2)
                     +- Scan csv  (1)


(1) Scan csv 
Output [3]: [category#211, status#216, amount#217]
Batched: false
Location: InMemoryFileIndex [file:/home/claude/Week6-Apache-Spark/data/source.csv]
PushedFilters: [IsNotNull(status), EqualTo(status,Completed)]
ReadSchema: struct<category:string,status:string,amount:double>

(2) Filter
Input [3]: [category#211, status#216, amount#217]
Condition : (isnotnull(status#216) AND (status#216 = Completed))

(3) Project
Output [2]: [category#211, amount#217]
Input [3]: [category#211, status#216, amount#217]

(4) HashAggregate
Input [2]: [category#211, amount#217]
Keys [1]: [category#211]
Functions [1]: [partial_sum(amount#217)]
Aggregate Attributes [1]: [sum#321]
Resu

---
## Q8 — Filter with AND condition

**Question:** Filter a DataFrame `df_orders` where `status = "Completed"` AND `amount > 1000`.


In [13]:
# Using & for AND between two column conditions (parentheses required around each condition)
completed_high_value = df_orders.filter((F.col("status") == "Completed") & (F.col("amount") > 1000))

print(f"Matching rows: {completed_high_value.count()}")
completed_high_value.select("order_id", "product_name", "status", "amount").show(10)


Matching rows: 11


+--------+-----------------+---------+-------+
|order_id|     product_name|   status| amount|
+--------+-----------------+---------+-------+
|    1001|   Wireless Mouse|Completed| 1198.0|
|    1003|Bluetooth Speaker|Completed| 1999.0|
|    1005|    Gaming Laptop|Completed|75000.0|
|    1009|      Smart Watch|Completed| 8999.0|
|    1011|       4K Monitor|Completed|18999.0|
|    1013|         Keyboard|Completed| 2598.0|
|    1015|     External SSD|Completed| 6499.0|
|    1016|      Office Desk|Completed| 7800.0|
|    1017|       Power Bank|Completed| 2998.0|
|    1019|           Tablet|Completed|22999.0|
+--------+-----------------+---------+-------+
only showing top 10 rows



---
## Q9 — Predicate Pushdown in Parquet

**Question:** Explain Predicate Pushdown in Parquet and its impact on memory usage and performance.

### Explanation

**Predicate Pushdown** is an optimization where Spark pushes a `WHERE`/`filter` condition **down to the data source/file format reader itself**, instead of reading all the data first and filtering it afterward in memory.

For Parquet specifically:

- Each Parquet file is divided into **row groups**, and each row group stores **column-level statistics** (min, max, null count) in its footer metadata.
- When a filter like `amount > 1000` is pushed down, Spark's Parquet reader checks these min/max statistics **before** decoding the actual data — if a row group's max value for `amount` is ≤ 1000, that **entire row group is skipped** without ever being read or deserialized.
- Within a row group, dictionary-encoded columns can also be filtered using their dictionaries before full decompression.

**Impact:**

- **Performance**: Drastically reduces the volume of data physically read from disk/object storage (fewer I/O operations), and reduces CPU spent decompressing/deserializing data that would be discarded anyway.
- **Memory usage**: Less data needs to be materialized into JVM memory/Spark's in-memory columnar format, lowering memory pressure and the chance of spills/OOM on large filters.

Predicate pushdown is automatically enabled in Spark for Parquet (`spark.sql.parquet.filterPushdown = true` by default) and works best when filters are simple comparisons on columns (not on derived/UDF expressions).


In [14]:
# Confirm pushdown config and show it in the physical plan as 'PushedFilters'
print("Parquet filter pushdown enabled:", spark.conf.get("spark.sql.parquet.filterPushdown"))

# Write a small parquet sample to demonstrate, then read + filter it
sample_parquet_path = "../data/sample_parquet"
shutil.rmtree(sample_parquet_path, ignore_errors=True)
df_orders.write.mode("overwrite").parquet(sample_parquet_path)

df_parquet_sample = spark.read.parquet(sample_parquet_path)
filtered_parquet = df_parquet_sample.filter(F.col("amount") > 1000)

print("\nPhysical plan — note the 'PushedFilters' entry on the Parquet scan:")
filtered_parquet.explain(mode="simple")

filtered_parquet.select("order_id", "amount").show(5)


Parquet filter pushdown enabled: true



Physical plan — note the 'PushedFilters' entry on the Parquet scan:
== Physical Plan ==
*(1) Filter (isnotnull(amount#403) AND (amount#403 > 1000.0))
+- *(1) ColumnarToRow
   +- FileScan parquet [order_id#394,product_id#395,product_name#396,category#397,old_name#398,price#399,base_price#400,quantity#401,status#402,amount#403,region#404,priority#405,user_id#406] Batched: true, DataFilters: [isnotnull(amount#403), (amount#403 > 1000.0)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/claude/Week6-Apache-Spark/data/sample_parquet], PartitionFilters: [], PushedFilters: [IsNotNull(amount), GreaterThan(amount,1000.0)], ReadSchema: struct<order_id:int,product_id:string,product_name:string,category:string,old_name:string,price:s...




+--------+-------+
|order_id| amount|
+--------+-------+
|    1001| 1198.0|
|    1002| 4500.0|
|    1003| 1999.0|
|    1005|75000.0|
|    1006| 3200.0|
+--------+-------+
only showing top 5 rows



---
## Q10 — Add a Derived Column

**Question:** Add a new column `final_price` equal to `base_price × 1.18`.


In [15]:
df_with_final_price = df_orders.withColumn("final_price", F.round(F.col("base_price") * 1.18, 2))

df_with_final_price.select("order_id", "product_name", "base_price", "final_price").show(10)


+--------+-----------------+----------+-----------+
|order_id|     product_name|base_price|final_price|
+--------+-----------------+----------+-----------+
|    1001|   Wireless Mouse|     500.0|      590.0|
|    1002|     Office Chair|    4000.0|     4720.0|
|    1003|Bluetooth Speaker|    1700.0|     2006.0|
|    1004|     Notebook Set|     120.0|      141.6|
|    1005|    Gaming Laptop|   65000.0|    76700.0|
|    1006|      Study Table|    2800.0|     3304.0|
|    1007|      USB-C Cable|     250.0|      295.0|
|    1008|        Desk Lamp|     750.0|      885.0|
|    1009|      Smart Watch|    7800.0|     9204.0|
|    1010|      Sketch Pens|      80.0|       94.4|
+--------+-----------------+----------+-----------+
only showing top 10 rows



---
## Q11 — Transformations vs Actions

**Question:** Explain the difference between Transformations and Actions. Provide at least two examples of each.

### Explanation

| | Transformations | Actions |
|---|---|---|
| **Definition** | Operations that produce a **new DataFrame/RDD** from an existing one. **Lazy** — only build up the logical plan. | Operations that **trigger execution** of the built-up plan and return a result to the driver or write data to storage. |
| **Execution** | Not executed immediately. | Executed immediately — this is what kicks off an actual Spark job. |
| **Sub-types** | **Narrow** (no shuffle: `filter`, `select`, `withColumn`, `map`) vs **Wide** (shuffle required: `groupBy`, `join`, `distinct`, `orderBy`) — see Q12. | N/A |
| **Examples** | `select()`, `filter()`/`where()`, `withColumn()`, `groupBy()`, `join()`, `orderBy()`, `distinct()`, `withColumnRenamed()` | `show()`, `count()`, `collect()`, `write()`, `take()`, `first()`, `head()` |

Two transformation examples and two action examples are demonstrated below.


In [16]:
# --- Transformations (lazy, build plan only) ---
t1 = df_orders.filter(F.col("category") == "Furniture")     # transformation
t2 = t1.select("order_id", "product_name", "amount")         # transformation

print("Two transformations chained — still no job has executed.")
print(type(t2))

# --- Actions (trigger execution) ---
row_count = t2.count()          # action #1
print(f"\nAction .count() executed -> {row_count} rows")

t2.show(5)                      # action #2


Two transformations chained — still no job has executed.
<class 'pyspark.sql.dataframe.DataFrame'>



Action .count() executed -> 6 rows


+--------+------------+------+
|order_id|product_name|amount|
+--------+------------+------+
|    1002|Office Chair|4500.0|
|    1006| Study Table|3200.0|
|    1008|   Desk Lamp|1798.0|
|    1012|   Bookshelf|2499.0|
|    1016| Office Desk|7800.0|
+--------+------------+------+
only showing top 5 rows



---
## Q12 — Build a Mini Pipeline: Load Parquet → Drop Nulls → Write CSV

**Question:** Load a Parquet file from `path/to/input`, remove rows where `user_id` is null, and save the output as CSV in `path/to/output`.

We use the Parquet sample written during Q9 (`../data/sample_parquet`) as our `path/to/input`, and write the cleaned result to `../output/csv` as `path/to/output` — matching the assignment's repository structure.


In [17]:
input_path = "../data/sample_parquet"      # path/to/input
output_path = "../output/csv/q12_clean_orders"  # path/to/output

# 1. Load Parquet
df_input = spark.read.parquet(input_path)
print(f"Rows before null removal: {df_input.count()}")
print(f"Rows with NULL user_id  : {df_input.filter(F.col('user_id').isNull()).count()}")

# 2. Remove rows where user_id is null  (.na.drop is the idiomatic Spark API for this)
df_clean = df_input.na.drop(subset=["user_id"])
print(f"Rows after null removal : {df_clean.count()}")

# 3. Save as CSV
shutil.rmtree(output_path, ignore_errors=True)
df_clean.write.mode("overwrite").option("header", "true").csv(output_path)

print(f"\nCleaned data written as CSV to: {output_path}")
df_clean.select("order_id", "user_id", "status").show(10)


Rows before null removal: 20


Rows with NULL user_id  : 3


Rows after null removal : 17



Cleaned data written as CSV to: ../output/csv/q12_clean_orders
+--------+-------+---------+
|order_id|user_id|   status|
+--------+-------+---------+
|    1001|   U001|Completed|
|    1002|   U002|  Pending|
|    1003|   U003|Completed|
|    1004|   U004|Completed|
|    1005|   U005|Completed|
|    1007|   U007|Completed|
|    1008|   U008|  Pending|
|    1009|   U009|Completed|
|    1010|   U010|Completed|
|    1011|   U011|Completed|
+--------+-------+---------+
only showing top 10 rows



### Handling Nulls Efficiently — Additional Patterns

Beyond simply dropping nulls, Spark provides several efficient, vectorized ways to handle missing values without ever pulling data to the driver:


In [18]:
# Pattern 1: drop rows with ANY null across all columns
df_orders.na.drop(how="any").count()

# Pattern 2: drop rows only if ALL columns are null
df_orders.na.drop(how="all").count()

# Pattern 3: fill nulls with a default value (column-specific)
df_filled = df_orders.na.fill({"user_id": "UNKNOWN"})
print("Null user_id values replaced with 'UNKNOWN':")
df_filled.filter(F.col("order_id").isin(1006, 1012, 1018)).select("order_id", "user_id").show()

# Pattern 4: count nulls per column efficiently (single pass, no collect() of raw data)
null_counts = df_orders.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_orders.columns
])
null_counts.show()


Null user_id values replaced with 'UNKNOWN':


+--------+-------+
|order_id|user_id|
+--------+-------+
|    1006|UNKNOWN|
|    1012|UNKNOWN|
|    1018|UNKNOWN|
+--------+-------+



+--------+----------+------------+--------+--------+-----+----------+--------+------+------+------+--------+-------+
|order_id|product_id|product_name|category|old_name|price|base_price|quantity|status|amount|region|priority|user_id|
+--------+----------+------------+--------+--------+-----+----------+--------+------+------+------+--------+-------+
|       0|         0|           0|       0|       0|    0|         0|       0|     0|     0|     0|       0|      3|
+--------+----------+------------+--------+--------+-----+----------+--------+------+------+------+--------+-------+



---
## CSV vs Parquet — Performance Comparison (Q4 Continued)

To make the Q4 comparison concrete, we write the same dataset in both formats and compare **file size on disk** and **read+filter time**. (Our sample dataset is intentionally small for demo purposes — on real multi-GB/TB datasets the gap is far more pronounced.)


In [19]:
csv_out = "../output/csv/comparison_orders"
parquet_out = "../output/parquet/comparison_orders"

shutil.rmtree(csv_out, ignore_errors=True)
shutil.rmtree(parquet_out, ignore_errors=True)

df_orders.write.mode("overwrite").option("header", "true").csv(csv_out)
df_orders.write.mode("overwrite").parquet(parquet_out)

def dir_size_bytes(path):
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total

csv_size = dir_size_bytes(csv_out)
parquet_size = dir_size_bytes(parquet_out)

print(f"CSV folder size      : {csv_size} bytes")
print(f"Parquet folder size  : {parquet_size} bytes")
print(f"Parquet is {csv_size / parquet_size:.2f}x smaller on this sample" if parquet_size else "n/a")


CSV folder size      : 1985 bytes
Parquet folder size  : 4816 bytes
Parquet is 0.41x smaller on this sample


In [20]:
# Read + filter timing comparison (illustrative on this small sample)
start = time.time()
spark.read.option("header", "true").option("inferSchema", "true").csv(csv_out).filter(F.col("amount") > 1000).count()
csv_time = time.time() - start

start = time.time()
spark.read.parquet(parquet_out).filter(F.col("amount") > 1000).count()
parquet_time = time.time() - start

print(f"CSV     read+filter time : {csv_time:.4f} sec")
print(f"Parquet read+filter time : {parquet_time:.4f} sec")
print("\nNote: on small local data, JVM/JIT warm-up dominates timings;")
print("the *structural* advantages of Parquet (schema, compression, pushdown, column pruning)")
print("are what produce large, consistent gains on real multi-GB/TB production datasets.")


CSV     read+filter time : 0.4996 sec
Parquet read+filter time : 0.3261 sec

Note: on small local data, JVM/JIT warm-up dominates timings;
the *structural* advantages of Parquet (schema, compression, pushdown, column pruning)
are what produce large, consistent gains on real multi-GB/TB production datasets.


---
## Q13 — Client Mode vs Cluster Mode

**Question:** Explain the difference between Client Mode and Cluster Mode in Spark Architecture.

### Explanation

These refer to **where the Driver process runs** relative to the cluster, when submitting via `spark-submit --deploy-mode client|cluster`.

| | Client Mode | Cluster Mode |
|---|---|---|
| **Driver location** | Runs on the **machine that submits the job** (e.g., your laptop, an edge node, a notebook server) — outside the cluster's resource management. | Runs **inside the cluster**, as a process launched and managed by the Cluster Manager (e.g., a YARN container / K8s pod), alongside the Executors. |
| **Network dependency** | The submitting machine must stay connected/online for the **entire job duration** — if it disconnects or the laptop sleeps, the driver (and the whole application) dies. | No dependency on the submitting machine after submission — you can disconnect and the job keeps running on the cluster. |
| **Latency** | Driver↔Executor communication may cross a slower network link (submitting machine to cluster), adding latency to task scheduling. | Driver runs **on the same cluster network** as Executors — lower-latency communication. |
| **Typical use case** | Interactive workloads, debugging, **notebooks (like this one, via `local[*]`)**, development. | Production batch jobs, scheduled pipelines, long-running jobs submitted via orchestration tools (Airflow, etc.). |
| **Resource management** | Driver resources are *not* managed by the cluster manager. | Driver resources (memory/cores) are requested and managed by the cluster manager just like executors. |

This notebook itself runs effectively as a **local/client-style** deployment (`local[*]`) since the driver (this notebook's Python process) and the "executors" share the same machine — appropriate for development and the scope of this assignment.


In [21]:
# We can't switch deploy modes inside a running local notebook (deploy-mode is a spark-submit launch flag),
# but we can inspect how our current session is configured, which mirrors 'client mode' semantics:
print("deploy-mode equivalent for this session : client-style (driver = this notebook process)")
print("master                                  :", spark.sparkContext.master)
print("driver host                             :", spark.sparkContext.getConf().get("spark.driver.host"))


deploy-mode equivalent for this session : client-style (driver = this notebook process)
master                                  : local[*]
driver host                             : 192.0.2.2


---
## Q14 — Filter with OR condition

**Question:** Filter rows where `region = "North"` OR `priority = "High"`.


In [22]:
# Using | for OR between two column conditions
north_or_high_priority = df_orders.filter((F.col("region") == "North") | (F.col("priority") == "High"))

print(f"Matching rows: {north_or_high_priority.count()}")
north_or_high_priority.select("order_id", "region", "priority", "product_name").show(10)


Matching rows: 12
+--------+------+--------+-----------------+
|order_id|region|priority|     product_name|
+--------+------+--------+-----------------+
|    1001| North|    High|   Wireless Mouse|
|    1003| North|  Medium|Bluetooth Speaker|
|    1005|  West|    High|    Gaming Laptop|
|    1007| North|     Low|      USB-C Cable|
|    1009|  West|    High|      Smart Watch|
|    1010| North|     Low|      Sketch Pens|
|    1011| South|    High|       4K Monitor|
|    1013| North|  Medium|         Keyboard|
|    1015| South|    High|     External SSD|
|    1016| North|  Medium|      Office Desk|
+--------+------+--------+-----------------+
only showing top 10 rows



---
## Wide Transformations and Shuffle

**Explanation:**

- **Narrow transformations** (`filter`, `select`, `withColumn`, `map`): each output partition depends on **only one** input partition. No data needs to move between executors — Spark can execute these within a single stage, partition-by-partition, in parallel with no network cost.
- **Wide transformations** (`groupBy`, `join`, `distinct`, `orderBy`, `repartition`): each output partition can depend on **multiple** input partitions, because rows with the same key (e.g., the same `category` for a `groupBy`) might live on different executors. Spark must **shuffle** — redistribute data across the cluster (write to disk, transfer over network, read back) — so that all rows sharing a key land on the same partition/executor before the operation can complete.

**Why shuffle is expensive:** it involves disk I/O (writing shuffle files), network I/O (transferring data between executors), and serialization/deserialization — by far the most common source of slow Spark jobs. A wide transformation always creates a **new stage boundary** in the DAG.

**Best practice:** minimize the number of wide transformations, filter/select down to needed columns *before* a shuffle (reduces the amount of data shuffled), and pick sensible `spark.sql.shuffle.partitions` for your data size (too many → many tiny tasks; too few → skewed/large partitions).


In [23]:
# groupBy + agg is a WIDE transformation -> triggers a shuffle
revenue_by_category = (
    df_orders
    .groupBy("category")          # <-- shuffle boundary: rows with same category must co-locate
    .agg(
        F.count("order_id").alias("num_orders"),
        F.round(F.sum("amount"), 2).alias("total_revenue")
    )
    .orderBy(F.desc("total_revenue"))   # <-- ALSO a wide transformation (global sort)
)

print("Physical plan — look for 'Exchange' operators, which represent shuffle boundaries:")
revenue_by_category.explain(mode="simple")

revenue_by_category.show()


Physical plan — look for 'Exchange' operators, which represent shuffle boundaries:
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [total_revenue#1129 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(total_revenue#1129 DESC NULLS LAST, 8), ENSURE_REQUIREMENTS, [plan_id=907]
      +- HashAggregate(keys=[category#211], functions=[count(order_id#208), sum(amount#217)])
         +- Exchange hashpartitioning(category#211, 8), ENSURE_REQUIREMENTS, [plan_id=904]
            +- HashAggregate(keys=[category#211], functions=[partial_count(order_id#208), partial_sum(amount#217)])
               +- FileScan csv [order_id#208,category#211,amount#217] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/claude/Week6-Apache-Spark/data/source.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:int,category:string,amount:double>




+-----------+----------+-------------+
|   category|num_orders|total_revenue|
+-----------+----------+-------------+
|Electronics|        10|     142186.0|
|  Furniture|         6|      25197.0|
| Stationery|         4|       3636.0|
+-----------+----------+-------------+



---
## Q15 — Why `.show(5)` over `.collect()` on Multi-Terabyte Datasets

**Question:** Explain why `.show(5)` is preferred over `.collect()` while exploring multi-terabyte datasets.

### Explanation

- **`.collect()`** pulls the **entire** DataFrame's rows from all executors back to the **Driver's memory**, as a Python list. On a multi-terabyte dataset, this will either be extremely slow (massive network transfer to a single node) or **crash the driver with an OutOfMemoryError**, since the driver typically has far less memory than the whole cluster's executors combined. `collect()` should only ever be used on data you *know* is small (e.g., the result of an aggregation that's already been reduced to a handful of rows).
- **`.show(n)`** (default `n=20`) is an action that asks Spark to compute and display only the **first `n` rows**. Spark's query planner is often smart enough to **avoid scanning the entire dataset** to satisfy this — especially when combined with a `LIMIT`, Spark can stop processing partitions as soon as enough rows are found. Critically, the data is **formatted and printed directly to the console**, never materialized in the driver's memory as a full Python collection.

**Rule of thumb:** use `show()`, `take(n)`, `head(n)`, or `limit(n)` for exploration; only use `collect()` on a DataFrame that has already been filtered/aggregated down to a small, driver-safe size — or better yet, write the results to disk and inspect the output files instead.


In [24]:
# .show(5) -- safe for exploring any size of dataset
print("Using .show(5) — safe, bounded, never overwhelms driver memory:")
df_orders.show(5)

# Demonstrating the safe driver-side pattern when you DO need values in Python:
# only collect after the data has been reduced to something small (e.g., an aggregate)
small_summary = df_orders.groupBy("category").count()   # already reduced to a handful of rows
summary_rows = small_summary.collect()   # SAFE here because the row count is tiny (post-aggregation)
print("\nSafe use of .collect() — only after aggregating down to a few rows:")
for row in summary_rows:
    print(row)


Using .show(5) — safe, bounded, never overwhelms driver memory:
+--------+----------+-----------------+-----------+----------+-------+----------+--------+---------+-------+------+--------+-------+
|order_id|product_id|     product_name|   category|  old_name|  price|base_price|quantity|   status| amount|region|priority|user_id|
+--------+----------+-----------------+-----------+----------+-------+----------+--------+---------+-------+------+--------+-------+
|    1001|      P001|   Wireless Mouse|Electronics|Customer A|  599.0|     500.0|       2|Completed| 1198.0| North|    High|   U001|
|    1002|      P002|     Office Chair|  Furniture|Customer B| 4500.0|    4000.0|       1|  Pending| 4500.0| South|     Low|   U002|
|    1003|      P003|Bluetooth Speaker|Electronics|Customer C| 1999.0|    1700.0|       1|Completed| 1999.0| North|  Medium|   U003|
|    1004|      P004|     Notebook Set| Stationery|Customer D|  150.0|     120.0|       5|Completed|  750.0|  East|     Low|   U004|
|    


Safe use of .collect() — only after aggregating down to a few rows:
Row(category='Electronics', count=10)
Row(category='Furniture', count=6)
Row(category='Stationery', count=4)


---
## Full Data Pipeline: Read → Transform → Filter → Write

Bringing every concept together into one end-to-end production-style pipeline, following Spark best practices throughout (explicit schema, no `collect()` on big data, writing both CSV and Parquet outputs).


In [25]:
def run_orders_pipeline(spark, input_csv_path, output_dir):
    """
    Read -> Transform -> Filter -> Write pipeline for the orders dataset.

    Steps:
      1. READ      : Load CSV with an explicit schema (single pass, no inferSchema overhead)
      2. TRANSFORM : Rename column, cast types, add a derived 'final_price' column
      3. FILTER    : Keep only Completed orders with a non-null user_id
      4. WRITE     : Persist the result in both CSV and Parquet formats
    """
    # 1. READ
    df = spark.read.csv(input_csv_path, header=True, schema=explicit_schema)

    # 2. TRANSFORM
    df_transformed = (
        df
        .withColumnRenamed("old_name", "customer_name")
        .withColumn("price", F.col("price").cast(DoubleType()))
        .withColumn("final_price", F.round(F.col("base_price") * 1.18, 2))
    )

    # 3. FILTER
    df_filtered = (
        df_transformed
        .filter(F.col("status") == "Completed")
        .na.drop(subset=["user_id"])
    )

    # 4. WRITE (both formats, as required by the assignment)
    csv_path = f"{output_dir}/csv/final_orders"
    parquet_path = f"{output_dir}/parquet/final_orders"

    shutil.rmtree(csv_path, ignore_errors=True)
    shutil.rmtree(parquet_path, ignore_errors=True)

    df_filtered.write.mode("overwrite").option("header", "true").csv(csv_path)
    df_filtered.write.mode("overwrite").parquet(parquet_path)

    return df_filtered, csv_path, parquet_path


result_df, csv_path, parquet_path = run_orders_pipeline(
    spark,
    input_csv_path="../data/source.csv",
    output_dir="../output"
)

print(f"Pipeline complete. {result_df.count()} rows written.")
print(f"  CSV output     -> {csv_path}")
print(f"  Parquet output -> {parquet_path}")

result_df.select("order_id", "customer_name", "price", "final_price", "status").show(10)


Pipeline complete. 14 rows written.
  CSV output     -> ../output/csv/final_orders
  Parquet output -> ../output/parquet/final_orders
+--------+-------------+-------+-----------+---------+
|order_id|customer_name|  price|final_price|   status|
+--------+-------------+-------+-----------+---------+
|    1001|   Customer A|  599.0|      590.0|Completed|
|    1003|   Customer C| 1999.0|     2006.0|Completed|
|    1004|   Customer D|  150.0|      141.6|Completed|
|    1005|   Customer E|75000.0|    76700.0|Completed|
|    1007|   Customer G|  299.0|      295.0|Completed|
|    1009|   Customer I| 8999.0|     9204.0|Completed|
|    1010|   Customer J|   99.0|       94.4|Completed|
|    1011|   Customer K|18999.0|    19470.0|Completed|
|    1013|   Customer M| 1299.0|     1298.0|Completed|
|    1015|   Customer O| 6499.0|     6608.0|Completed|
+--------+-------------+-------+-----------+---------+
only showing top 10 rows



---
## Spark Best Practices & Optimization Tips Applied in This Notebook

1. **Avoid `collect()` on large data** — use `show()`, `take()`, `limit()`; only `collect()` after aggregating down to a small result (Q15).
2. **Prefer explicit schemas over `inferSchema`** for production pipelines — avoids an extra read pass and guarantees type correctness (Q3).
3. **Prefer Parquet over CSV** for intermediate/internal storage — columnar layout, compression, predicate pushdown, and column pruning all reduce I/O (Q4, Q9).
4. **Filter early, select only needed columns** — reduces the amount of data carried through later (especially wide) transformations and shuffles.
5. **Minimize wide transformations** (`groupBy`, `join`, `orderBy`, `distinct`) and be deliberate about `spark.sql.shuffle.partitions` for your data size, since these trigger expensive shuffles.
6. **Handle nulls explicitly and early** (`na.drop`, `na.fill`) rather than letting them silently propagate into downstream aggregations/joins.
7. **Use `.explain()`** to inspect logical/physical plans and confirm optimizations (pushdown, pruning) are actually being applied.
8. **Cache/persist** intermediate DataFrames only when they are reused multiple times in the DAG — caching everything wastes memory.
9. **Write partitioned, columnar output** (Parquet) as the default pipeline output format; only emit CSV when an external system specifically requires it.


---
## Conclusion

This notebook covered the full breadth of Week 6's objectives: how Spark applications are architected and executed (Driver / Cluster Manager / Executor, client vs cluster deploy mode), how Spark's **lazy evaluation** and **DAG/lineage** model enables both query optimization and fault tolerance, and the practical DataFrame API for **reading, filtering, transforming, and writing** data.

We also worked through the performance-critical concepts of **wide transformations and shuffle**, **predicate pushdown**, and a hands-on **CSV vs Parquet** comparison — reinforcing *why* columnar formats are the default choice for analytical pipelines at scale. Finally, all of this was assembled into a single, reusable **Read → Transform → Filter → Write pipeline function**, written using only best-practice patterns (explicit schemas, no unsafe `collect()`, dual CSV/Parquet output).

These foundations — architecture awareness, lazy/DAG-based execution, and disciplined DataFrame transformations — are exactly what's needed to write Spark jobs that scale safely from a 20-row sample dataset like this one up to multi-terabyte production data.


In [26]:
# Clean shutdown of the Spark session
spark.stop()
print("SparkSession stopped.")


SparkSession stopped.
